# Ensemble Learning

## Project Overview

### Introduction
This notebook serves as an exploration of **Ensemble Learning** techniques where multiple models (often called "weak learners") are combined to solve a single problem. The goal is to produce a "strong learner" that is more accurate than any individual component.

### Current Scope (Work in Progress)

1.  **Voting Classifiers**: Combining diverse algorithms (Logistic Regression, SVM, Random Forests) to make predictions based on majority consensus.
2.  **Bagging & Pasting**: Reducing variance by training predictors on random subsets of the training data (e.g., Random Forests).
3.  **Boosting (AdaBoost)**: A sequential technique that focuses on correcting the errors of previous models, particularly useful for complex, imbalanced datasets.

### Business Case Study: HR Attrition Prediction
We apply the **AdaBoost** algorithm to the **IBM HR Analytics Attrition Dataset**. From a business perspective, this analysis aims to:
*   **Identify High-Risk Turnover**: Detect employees likely to leave, enabling proactive retention strategies.


## Voting Classifiers

make_moons generates 2D binary classification dataset. make_moons produces two interleaving half-circles.  
[sklearn docs](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html)

In [2]:
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

In [3]:
X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)


VotingClassifier is sklearn class that receives a list of predictors and acts like a normal classifier.

In [4]:
voting_clf = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(random_state=42)),
        ('rf', RandomForestClassifier(random_state=42)),
        ('svc', SVC(random_state=42))]
)
voting_clf.fit(X_train, y_train)

VotingClassifier(estimators=[('lr', LogisticRegression(random_state=42)),
                             ('rf', RandomForestClassifier(random_state=42)),
                             ('svc', SVC(random_state=42))])

It clones and fits every estimator. Original scores can be accessed via named_estimators_ attribute.

In [5]:
for name, clf in voting_clf.named_estimators_.items():
    print(name, clf.score(X_test, y_test))

lr 0.864
rf 0.896
svc 0.896


Call predict() method to perform hard voting (it outputs 1 as a class if 2 out of 3 classifiers predicted this class).

In [6]:
voting_clf.predict(X_test[:1])

array([1])

In [7]:
[clf.predict(X_test[:1]) for clf in voting_clf.estimators_]

[array([1]), array([1]), array([0])]

In [8]:
voting_clf.score(X_test, y_test)

0.912

Voting classifier outperforms all the individual classifiers. It is a reason to use an ansemble of weak learners if they are sufficiently diverse. Theoretically, they focus on different aspects of the data and make different errors. But this is true only if classifiers are independent.

What is the difference between soft and hard voting?
If classifier is able to estimate class probability, then VotingClassifier predict the class with the higher probability averaged across all individual classifiers.

## Bagging and Pasting

Train the same algorithm on different random subset of data. Bagging - with replacement, pasting - without.   
Once all predictors are trained, the ensemble aggregates the individual predictons of all individual predictors.

In [9]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag_clf = BaggingClassifier(
    DecisionTreeClassifier(), n_estimators=500,
    max_samples=100, n_jobs=-1
)
bag_clf.fit(X_train, y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), max_samples=100,
                  n_estimators=500, n_jobs=-1)

### Random Forests

It is a decision tree classifier with BaggingClassifier ensemble. It introduces extra randomness - it searches for best feature among random subset of features, instead of seraching for the best overall feature.

In [11]:
from sklearn.ensemble import RandomForestClassifier

rnd_clf = RandomForestClassifier(n_estimators=500, max_leaf_nodes=16, n_jobs=-1, random_state=42)
rnd_clf.fit(X_train, y_train)

y_pred_rf = rnd_clf.predict(X_test)

The equivalent (although it is less optimised):

In [13]:
bag_clf = BaggingClassifier(
    DecisionTreeClassifier(max_features='sqrt', max_leaf_nodes=16),
    n_estimators=500, n_jobs=-1, random_state=42
)

#### Feature Importance



Random Forests make it easier to measure the relative importance of the feature. In sklearn this score computes automatically.

In [15]:
from sklearn.datasets import load_iris
iris = load_iris(as_frame=True)
rnd_clf = RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=42)
rnd_clf.fit(iris['data'], iris['target'])
for name, score in zip(iris['data'].columns, rnd_clf.feature_importances_):
    print(name, round(score, 2))

sepal length (cm) 0.11
sepal width (cm) 0.02
petal length (cm) 0.44
petal width (cm) 0.42


Petal lenght is used for 44% while sepal width is only used in 2%.

## Boosting

The general idea of most boosting methods is to train predictors sequentially, correcting predecessor.

### AdaBoost

An AdaBoost classifier is a meta-estimator that begins by fitting a classifier on the original dataset and then fits additional copies of the classifier on the same dataset but where the weights of incorrectly classified instances are adjusted such that subsequent classifiers focus more on difficult cases.

When to use it:
Small to medium sized, low-noise, structured datasets where interpretability is helpful. Credit scoring, anomaly detection.


[Sklearn docs](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html)



We can use it with[ HR Analytics dataset](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset?resource=download) since it contains imbalanced data - more employees stay than leave. AdaBoost's sequential learning focuses on previously misclassified instances, helping the model capture the characteristics of the minority class (those who leave).

In [72]:
from google.colab import userdata
import os
import kagglehub
import pandas as pd

os.environ["KAGGLE_API_TOKEN"] = userdata.get('KAGGLE_API_TOKEN')
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')

path = kagglehub.dataset_download("pavansubhasht/ibm-hr-analytics-attrition-dataset")

# List files in the downloaded directory to find the CSV
files = os.listdir(path)
print(f"Files in directory: {files}")

csv_path = os.path.join(path, 'WA_Fn-UseC_-HR-Employee-Attrition.csv')
df = pd.read_csv(csv_path)


Using Colab cache for faster access to the 'ibm-hr-analytics-attrition-dataset' dataset.
Files in directory: ['WA_Fn-UseC_-HR-Employee-Attrition.csv']


In [78]:
# Data Processing

df.drop(['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours'], axis="columns", inplace=True)

df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

dummy_col = ['BusinessTravel', 'Department', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime', 'EducationField']

# Apply One-Hot Encoding
df = pd.get_dummies(df, columns=dummy_col, drop_first=True)



In [81]:
from sklearn.model_selection import train_test_split

X = df.drop('Attrition', axis="columns")
y = df['Attrition']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state = 42)

In [82]:
from sklearn.ensemble import AdaBoostClassifier

ada_clf = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=1),
    n_estimators=30,
    random_state=42,
    learning_rate = 0.5)

ada_clf.fit(X_train, y_train)

AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                   learning_rate=0.5, n_estimators=30, random_state=42)

In [83]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, roc_auc_score

def evaluate(model, X_train, X_test, y_train, y_test):
    y_test_pred = model.predict(X_test)
    y_train_pred = model.predict(X_train)

    print("TRAINIG RESULTS: \n===============================")
    clf_report = pd.DataFrame(classification_report(y_train, y_train_pred, output_dict=True))
    print(f"CONFUSION MATRIX:\n{confusion_matrix(y_train, y_train_pred)}")
    print(f"ACCURACY SCORE:\n{accuracy_score(y_train, y_train_pred):.4f}")
    print(f"CLASSIFICATION REPORT:\n{clf_report}")

    print("TESTING RESULTS: \n===============================")
    clf_report = pd.DataFrame(classification_report(y_test, y_test_pred, output_dict=True))
    print(f"CONFUSION MATRIX:\n{confusion_matrix(y_test, y_test_pred)}")
    print(f"ACCURACY SCORE:\n{accuracy_score(y_test, y_test_pred):.4f}")
    print(f"CLASSIFICATION REPORT:\n{clf_report}")

In [84]:
evaluate(ada_clf, X_train, X_test, y_train, y_test)

TRAINIG RESULTS: 
CONFUSION MATRIX:
[[850   3]
 [132  44]]
ACCURACY SCORE:
0.8688
CLASSIFICATION REPORT:
                    0           1  accuracy    macro avg  weighted avg
precision    0.865580    0.936170  0.868805     0.900875      0.877654
recall       0.996483    0.250000  0.868805     0.623242      0.868805
f1-score     0.926431    0.394619  0.868805     0.660525      0.835470
support    853.000000  176.000000  0.868805  1029.000000   1029.000000
TESTING RESULTS: 
CONFUSION MATRIX:
[[380   0]
 [ 59   2]]
ACCURACY SCORE:
0.8662
CLASSIFICATION REPORT:
                    0          1  accuracy   macro avg  weighted avg
precision    0.865604   1.000000  0.866213    0.932802      0.884194
recall       1.000000   0.032787  0.866213    0.516393      0.866213
f1-score     0.927961   0.063492  0.866213    0.495726      0.808386
support    380.000000  61.000000  0.866213  441.000000    441.000000
